In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# 1. 读取之前保存的 clean 数据
print("正在加载猪单细胞数据...")
adata = sc.read_h5ad('adata_ec_clean2.h5ad')
print(f"数据加载成功！共包含 {adata.n_obs} 个细胞，{adata.n_vars} 个基因。")
print("内皮细胞亚群:", adata.obs['Final_Cell_Type'].unique())


In [ ]:
# =================【用户修改区域】=================
# 请在此处指定你数据中代表时间点的列名，例如 'Day', 'Timepoint', 'Group' 等
timepoint_col = 'time_point' 
# =================================================

# 检查列名是否存在
if timepoint_col not in adata.obs.columns:
    print(f"警告: '{timepoint_col}' 不在 adata.obs 中。当前可用的列有: {list(adata.obs.columns)}")
    print("请将 timepoint_col 修改为正确的列名后再运行下方代码。")

In [ ]:
# 2. 定义全面且系统的 9 大内皮功能基因集 (统一采用猪常用大写基因名)
comprehensive_ec_pathways = {
    # 血管生成与出芽 (Angiogenesis & Sprouting)
    'Angiogenesis': [
        'VEGFA', 'KDR', 'FLT1', 'FLT4', 'ANGPT2', 'DLL4', 'APLN', 'NRP1', 'FGFR1'
    ],
    # 内皮屏障与细胞紧密连接 (Barrier & Junctions - 维持血管不渗漏)
    'Vascular_Barrier': [
        'CDH5', 'CLDN5', 'TJP1', 'OCLN', 'F11R', 'ESAM', 'PECAM1'
    ],
    # 炎症激活与免疫细胞粘附 (Inflammatory Activation)
    'Inflammatory_Activation': [
        'SELE', 'SELP', 'VCAM1', 'ICAM1', 'CCL2', 'CX3CL1', 'IL6', 'NFKB1'
    ],
    # 机械血流剪切力响应与血管舒张 (Shear Stress & Vasodilation)
    'Shear_Stress_Response': [
        'KLF2', 'KLF4', 'NOS3', 'PTGIS', 'EDN1', 'MEF2C', 'PLAT'
    ],
    # 细胞外基质重塑 (ECM Remodeling)
    'ECM_Remodeling': [
        'COL4A1', 'COL4A2', 'FN1', 'LAMA4', 'NID1', 'MMP2', 'TIMP1', 'BGN'
    ],
    # 内皮间质转化 (EndMT - 内皮细胞向纤维化方向转变)
    'EndMT_Signature': [
        'SNAI1', 'SNAI2', 'TGFB1', 'TGFB2', 'ACTA2', 'TAGLN', 'COL1A1', 'FN1'
    ],
    # 内皮高糖酵解与脂质代谢 (Endothelial Metabolism)
    'EC_Metabolism': [
        'PFKFB3', 'LDHA', 'FABP4', 'LPL', 'CD36', 'SLC2A1'
    ],
    # 抗凝与血液稳态 (Anticoagulation & Hemostasis)
    'Anticoagulation': [
        'THBD', 'PROCR', 'PLAT', 'VWF', 'TFPI'
    ],
    # 缺氧响应 (Hypoxia Response)
    'Hypoxia_Response': [
        'HIF1A', 'VEGFA', 'LDHA', 'PDK1', 'SLC2A1', 'ADM'
    ]
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr, spearmanr
import seaborn as sns

# 0. 定义实际匹配成功的 8 个猪内皮模块
detected_ec_pathways = {
    'Angiogenesis': ['KDR', 'FLT1', 'FLT4', 'ANGPT2', 'DLL4', 'NRP1'],
    'Vascular_Barrier': ['CDH5', 'PECAM1'],
    'Inflammatory_Activation': ['SELE', 'SELP', 'VCAM1', 'ICAM1', 'CCL2'],
    'Shear_Stress_Response': [
        'KLF2',
        'KLF4',
        'NOS3',
        'PTGIS',
        'EDN1',
        'MEF2C',
        'PLAT',
    ],
    'ECM_Remodeling': ['COL4A1', 'FN1', 'LAMA4', 'NID1', 'MMP2', 'TIMP1', 'BGN'],
    'EndMT_Signature': ['TGFB1', 'TGFB2', 'ACTA2', 'TAGLN', 'COL1A1', 'FN1'],
    'EC_Metabolism': ['PFKFB3', 'LPL', 'CD36'],
    'Anticoagulation': ['PLAT', 'VWF', 'TFPI'],
}

# 【注意】请确保这里的 adata_ec 是你自己的内皮细胞对象变量名
for mod_name, genes in detected_ec_pathways.items():
  valid_genes = [g for g in genes if g in adata_ec.var_names]
  sc.tl.score_genes(
      adata_ec,
      gene_list=valid_genes,
      score_name=mod_name,
      ctrl_size=50,
      random_state=42,
  )

# 1. 计算 8 个模块的 Jaccard 重叠指数矩阵 (Panel A)
modules = list(detected_ec_pathways.keys())
n_mod = len(modules)
jaccard_mat = np.zeros((n_mod, n_mod))

for i in range(n_mod):
  for j in range(n_mod):
    set_i = set(detected_ec_pathways[modules[i]])
    set_j = set(detected_ec_pathways[modules[j]])
    jaccard_mat[i, j] = len(set_i.intersection(set_j)) / len(
        set_i.union(set_j)
    )

jaccard_df = pd.DataFrame(jaccard_mat, index=modules, columns=modules)

# 2. 计算细胞层面的模块得分共线性矩阵 (Panel B)
corr_df = adata_ec.obs[modules].corr(method='pearson')

# 3. 剔除重叠基因 (FN1, PLAT) 后的去重稳健性测试 (Panel C)
endmt_no_fn1 = [
    g
    for g in ['TGFB1', 'TGFB2', 'ACTA2', 'TAGLN', 'COL1A1']
    if g in adata_ec.var_names
]
sc.tl.score_genes(
    adata_ec,
    gene_list=endmt_no_fn1,
    score_name='EndMT_no_FN1',
    ctrl_size=50,
    random_state=42,
)

anticoag_no_plat = [g for g in ['VWF', 'TFPI'] if g in adata_ec.var_names]
sc.tl.score_genes(
    adata_ec,
    gene_list=anticoag_no_plat,
    score_name='Anticoag_no_PLAT',
    ctrl_size=50,
    random_state=42,
)

# 4. 无参排序评分法交叉验证 (Panel D)
expr_df = (
    adata_ec.raw.to_adata() if adata_ec.raw is not None else adata_ec
).to_df()
ranked_expr = expr_df.rank(axis=1, pct=True)

rank_corrs = {}
for mod_name, genes in detected_ec_pathways.items():
  valid_genes = [g for g in genes if g in ranked_expr.columns]
  rank_score = ranked_expr[valid_genes].mean(axis=1)
  r_val, _ = spearmanr(adata_ec.obs[mod_name], rank_score)
  rank_corrs[mod_name] = r_val

rank_corr_series = pd.Series(rank_corrs)

# 仅需重新运行第 5 步绘图部分（无需重算数据，秒出图）：
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# Panel A
sns.heatmap(
    jaccard_df,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0,
    vmax=0.3,
    linewidths=0.5,
    ax=axes[0, 0],
    cbar_kws={'label': 'Jaccard Similarity'},
)
axes[0, 0].set_title(
    'A. Pairwise Gene Overlap (Mean Jaccard = 0.007)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[0, 0].tick_params(axis='x', rotation=35)

# Panel B
sns.heatmap(
    corr_df,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=axes[0, 1],
    cbar_kws={'label': 'Pearson r'},
)
axes[0, 1].set_title(
    'B. Cell-Level Module Score Collinearity Matrix (Max |r| = 0.27)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[0, 1].tick_params(axis='x', rotation=35)

# Panel C (先画紫色的 Anticoagulation，再画橙色的 EndMT，防止遮挡)
sns.scatterplot(
    x=adata_ec.obs['Anticoagulation'],
    y=adata_ec.obs['Anticoag_no_PLAT'],
    s=10,
    alpha=0.25,
    color='#7570b3',
    label=f'Anticoagulation (without PLAT): r = {r_plat:.3f}',
    ax=axes[1, 0],
)
sns.scatterplot(
    x=adata_ec.obs['EndMT_Signature'],
    y=adata_ec.obs['EndMT_no_FN1'],
    s=12,
    alpha=0.45,
    color='#d95f02',
    label=f'EndMT (without FN1): r = {r_fn1:.3f}',
    ax=axes[1, 0],
)
axes[1, 0].plot([-1, 2.5], [-1, 2.5], ls='--', color='#333333', lw=1.2)
axes[1, 0].set_title(
    'C. De-duplication Sensitivity Analysis',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[1, 0].set_xlabel('Original Module Score (with shared gene)', fontsize=11)
axes[1, 0].set_ylabel(
    'De-duplicated Module Score (without shared gene)', fontsize=11
)
axes[1, 0].legend(loc='upper left', frameon=True, fontsize=10.5)

# Panel D (拉宽横轴到 1.15，并将图例放在左上角空白区或顶部，彻底避免遮挡 r = 0.95)
gene_counts = [len(detected_ec_pathways[m]) for m in modules]
bar_labels = [f'{m} (n={c})' for m, c in zip(modules, gene_counts)]

sns.barplot(
    x=rank_corr_series.values,
    y=bar_labels,
    color='#2b8cbe',
    ax=axes[1, 1],
    edgecolor='black',
    width=0.7,
)
axes[1, 1].axvline(
    0.8, ls='--', color='red', lw=1.5, label='Robustness threshold (r = 0.80)'
)
axes[1, 1].set_xlim(0, 1.15)
for idx, val in enumerate(rank_corr_series.values):
  axes[1, 1].text(
      val + 0.015, idx, f'r = {val:.2f}', va='center', fontsize=10, fontweight='bold'
  )

axes[1, 1].set_title(
    'D. Robustness to Gene Set Size (score_genes vs. Rank-based)',
    fontsize=12,
    fontweight='bold',
    pad=10,
)
axes[1, 1].set_xlabel(
    'Spearman Correlation (r) Across All Endothelial Cells', fontsize=11
)
axes[1, 1].legend(loc='lower left', frameon=True, fontsize=10)

plt.tight_layout()
plt.savefig('Supplementary_Fig_EC_Module_Robustness.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 自动适配数据中存在的基因
def get_existing_genes(gene_list, adata):
    return [g for g in gene_list if g in adata.var_names or g.upper() in adata.var_names]

# 3. 对 9 个模块进行打分
score_names = []
for pathway_name, genes in comprehensive_ec_pathways.items():
    valid_genes = get_existing_genes(genes, adata)
    if len(valid_genes) > 0:
        score_name = f'{pathway_name}_score'
        sc.tl.score_genes(adata, gene_list=valid_genes, score_name=score_name)
        score_names.append(score_name)
        print(f"成功为 【{pathway_name}】 打分，包含猪基因: {valid_genes}")
    else:
        print(f"警告：未在数据中找到 【{pathway_name}】 的任何基因，跳过。")

In [ ]:
# =================【用户修改区域】=================
timepoint_col = 'time_point' 
# =================================================

# 2. 提取评分列，同时【排除】包含 'doublet' 的列
score_cols = [col for col in adata.obs.columns if col.endswith('_score') and 'doublet' not in col.lower()]


In [ ]:
# 3. 计算 细胞亚群 + 时间点 的功能均值
df_scores = adata.obs[['Final_Cell_Type', timepoint_col] + score_cols]
mean_df = df_scores.groupby(['Final_Cell_Type', timepoint_col], observed=False).mean().reset_index()


In [ ]:
# 4. 纯Python字符提取数字（代替 re 模块，防止 NameError）
def extract_day_num(tp):
    # 提取字符串中的所有数字字符并拼合，例如 "D14" -> 14, "Day0" -> 0
    num_str = ''.join([c for c in str(tp) if c.isdigit()])
    return int(num_str) if num_str else 0

mean_df['time_numeric'] = mean_df[timepoint_col].apply(extract_day_num)
mean_df = mean_df.sort_values(by=['Final_Cell_Type', 'time_numeric'])


In [ ]:
# 5. 转换数据为长格式
melted_df = mean_df.melt(
    id_vars=['Final_Cell_Type', timepoint_col, 'time_numeric'], 
    value_vars=score_cols,
    var_name='Pathway', 
    value_name='Score'
)
# 简化通路名称
melted_df['Pathway'] = melted_df['Pathway'].str.replace('_score', '').str.replace('_', ' ')


In [ ]:
# 6. 绘图设置 (去除了 doublet 后有 8 个功能通路，画在 2x4 的画布上)
sns.set_theme(style="ticks")
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica'] 
plt.rcParams['axes.unicode_minus'] = False 

pathways = melted_df['Pathway'].unique()

# 创建 2x4 = 8个子图 的画布
fig, axes = plt.subplots(2, 4, figsize=(18, 9), dpi=150, sharex=True)
axes = axes.flatten()

# 循环绘制
for i, pathway in enumerate(pathways):
    ax = axes[i]
    pathway_data = melted_df[melted_df['Pathway'] == pathway]
    
    sns.lineplot(
        data=pathway_data,
        x=timepoint_col,
        y='Score',
        hue='Final_Cell_Type',
        marker='o', 
        linewidth=2.5,
        markersize=6,
        ax=ax,
        sort=False 
    )
    
    ax.set_title(pathway, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Score' if i % 4 == 0 else '') # 只在最左列显示 Y 轴标签
    ax.grid(True, linestyle='--', alpha=0.5) 
    ax.get_legend().remove()

# 如果实际通路数少于 8 个，删掉多余的空白子图
for j in range(len(pathways), len(axes)):
    fig.delaxes(axes[j])

# 统一底部图例
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, 
    labels, 
    loc='lower center', 
    bbox_to_anchor=(0.5, -0.08), 
    ncol=4, 
    fontsize=11, 
    frameon=True,
    title="Endothelial Cell Subtypes",
    title_fontsize=12
)

plt.suptitle('', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

plt.savefig('Pig_EC_Functional_Temporal_Dynamics_Lineplot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# [步骤 2-6 保持不变]
score_cols = [col for col in adata.obs.columns if col.endswith('_score') and 'doublet' not in col.lower()]
df_scores = adata.obs[['Final_Cell_Type'] + score_cols].copy()
for col in score_cols:
    df_scores[col] = pd.to_numeric(df_scores[col], errors='coerce')

df_melted = df_scores.melt(
    id_vars=['Final_Cell_Type'], 
    value_vars=score_cols,
    var_name='Pathway', 
    value_name='Score'
)
df_melted['Pathway'] = df_melted['Pathway'].str.replace('_score', '').str.replace('_', ' ')

summary = df_melted.groupby(['Final_Cell_Type', 'Pathway'], observed=False).agg(
    Mean_Score=('Score', 'mean'),
    Fraction_Positive=('Score', lambda x: (x > 0).sum() / len(x) * 100)
).reset_index()

# ================= 7. 转换为热图矩阵 =================
mean_matrix = summary.pivot(index='Final_Cell_Type', columns='Pathway', values='Mean_Score')
fraction_matrix = summary.pivot(index='Final_Cell_Type', columns='Pathway', values='Fraction_Positive')

# 格式化标注文字 (数值和百分比)
annot_labels = np.vectorize(lambda m, f: f"{m:.2f}\n({f:.0f}%)")(mean_matrix.values, fraction_matrix.values)

# ================= 8. 绘制大字体热图 =================
fig, ax = plt.subplots(figsize=(11, 8), dpi=200) # 👈 稍微调大画布

# 绘制热图
sns.heatmap(
    mean_matrix,
    annot=annot_labels,          
    fmt="",                      
    cmap="Blues",                
    linewidths=1.2,              # 稍微加粗网格线线宽
    linecolor='#E0E0E0',         
    ax=ax,
    cbar_kws={
        'shrink': 0.7,
        'aspect': 15
    },
    annot_kws={"size": 11, "weight": "bold"} # 👈 核心：格子字体放大到 11 且加粗
)

# ================= 9. 放大所有文本和标签 =================
# 1. 设置轴标签字体
ax.set_xlabel('', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('', 
             fontsize=15, fontweight='bold', pad=20)

# 2. 设置坐标轴刻度字体 (放大到 12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=12)

# 3. 设置右侧 Colorbar 字体
cbar = ax.collections[0].colorbar
cbar.set_label('Mean score of Pathway', fontsize=12, fontweight='bold', labelpad=10)
cbar.ax.tick_params(labelsize=11) # Colorbar 刻度值放大到 11

plt.tight_layout()
plt.savefig('Pig_EC_Comprehensive_Scores_Heatmap_LargeFont.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# 6. 保存计算后的 adata 到本地，防止进度丢失
# -------------------------------------------------------------
adata.write('adata_ec_clean2.h5ad', compression='gzip')
print("计算完评分的 adata 已成功保存至: adata_ec_clean.h5ad")
